In [106]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [107]:
train_df = pd.read_csv('../sign_lang_recognition/data/sign_mnist_train.csv')
test_df = pd.read_csv('../sign_lang_recognition/data/sign_mnist_test.csv')

train_labels = train_df['label'].values
train_images = train_df.drop('label', axis=1).values

test_labels = test_df['label'].values
test_images = test_df.drop('label', axis=1).values

In [108]:
train_imgs, val_imgs, train_lbls, val_lbls = train_test_split(
  train_images, train_labels,
  test_size=0.15,
  stratify=train_labels,
  random_state=42
)

In [109]:
class SignLanguageDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image = self.images[idx].reshape(28, 28).astype(np.uint8)
    label = torch.tensor(self.labels[idx], dtype=torch.long)

    if self.transform:
      image = self.transform(image)
    else:
      image = torch.tensor(image, dtype=torch.float32) / 255.0
      image = image.unsqueeze(0)

    return image, label

In [110]:
train_transform = transforms.Compose([
  transforms.ToPILImage(),
  transforms.RandomRotation(10),
  transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
  transforms.ToTensor(),
  transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
  transforms.ToPILImage(),
  transforms.ToTensor(),
  transforms.Normalize((0.5,), (0.5,))
])

In [111]:
train_dataset = SignLanguageDataset(train_imgs, train_lbls, transform=train_transform)
val_dataset = SignLanguageDataset(val_imgs, val_lbls, transform=test_transform)
test_dataset = SignLanguageDataset(test_images, test_labels, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [112]:
def train_model(model, epochs=20):
  optimizer = optim.Adam(model.parameters(), lr=0.001)
  loss_fn = nn.CrossEntropyLoss()

  best_val_acc = 0

  for epoch in range(epochs):
    model.train()
    train_correct, train_total = 0, 0

    for imgs, labels in train_loader:
      preds = model(imgs)
      loss = loss_fn(preds, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      train_correct += (preds.argmax(1) == labels).sum().item()
      train_total += labels.size(0)

    train_acc = train_correct / train_total

    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
      for imgs, labels in val_loader:
        preds = model(imgs)
        val_correct += (preds.argmax(1) == labels).sum().item()
        val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      best_weights = model.state_dict()

  model.load_state_dict(best_weights)
  print("Best Val Accuracy:", best_val_acc)

In [113]:
def test_model(model):

  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for imgs, labels in test_loader:
      outputs = model(imgs)
      predicted = outputs.argmax(dim=1)
      correct += (predicted == labels).sum().item()
      total += labels.size(0)

  accuracy = correct / total
  print("Test Accuracy:", accuracy)
  return accuracy

In [114]:
def count_params(model):
  return sum(p.numel() for p in model.parameters())


In [115]:
class BaseCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size = 3, padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    x = self.pool1(self.act1(self.conv1(x)))
    x = self.pool2(self.act2(self.conv2(x)))
    x = x.view(x.size(0), -1)
    x = self.fc2(self.act3(self.fc1(x)))

    return x


In [116]:
base_cnn = BaseCNN()
train_model(base_cnn)
base_cnn_test_accuracy = test_model(base_cnn)
print("CNN Parameters: ", count_params(base_cnn))

Epoch 1: Train=0.3575, Val=0.6480
Epoch 2: Train=0.6603, Val=0.8356
Epoch 3: Train=0.7837, Val=0.9136
Epoch 4: Train=0.8414, Val=0.9446
Epoch 5: Train=0.8734, Val=0.9648
Epoch 6: Train=0.8990, Val=0.9767
Epoch 7: Train=0.9131, Val=0.9757
Epoch 8: Train=0.9279, Val=0.9876
Epoch 9: Train=0.9371, Val=0.9888
Epoch 10: Train=0.9448, Val=0.9951
Epoch 11: Train=0.9485, Val=0.9964
Epoch 12: Train=0.9485, Val=0.9932
Epoch 13: Train=0.9553, Val=0.9917
Epoch 14: Train=0.9563, Val=0.9944
Epoch 15: Train=0.9620, Val=0.9925
Epoch 16: Train=0.9640, Val=0.9968
Epoch 17: Train=0.9623, Val=0.9973
Epoch 18: Train=0.9654, Val=0.9947
Epoch 19: Train=0.9678, Val=0.9971
Epoch 20: Train=0.9679, Val=0.9971
Best Val Accuracy: 0.9973294488953629
Test Accuracy: 0.9687674288901282
CNN Parameters:  27193


Although the Base CNN is able to reach a high validation accuracy, the test accuracy isn't as high which indicates that the model might not be able to generalize to unseen data as well. Additionally, training accuracy continues to increase across epochs, which suggests that the model could be learning dataset specific patterns. Because of this, I tried to introduce a dropout as a regularization technique. Since Dropout randomly deactivates neurons during training, it could help prevent the moodel from relying too much on specific features, which can help generalization and reduce any overfitting.

In [117]:
class DropoutCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)
    self.dropout1 = nn.Dropout2d(0.4)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)
    self.dropout2 = nn.Dropout2d(0.4)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    x = self.pool1(self.act1(self.conv1(x)))
    x = self.dropout1(x)
    x = self.pool2(self.act2(self.conv2(x)))
    x = self.dropout2(x)
    x = x.view(x.size(0), -1)
    x = self.fc2(self.act3(self.fc1(x)))

    return x

In [118]:
cnn_dropout = DropoutCNN()
train_model(cnn_dropout)
cnn_dropout_test_accuracy = test_model(cnn_dropout)
print("CNN Parameters: ", count_params(cnn_dropout))

Epoch 1: Train=0.2993, Val=0.6169
Epoch 2: Train=0.5162, Val=0.7759
Epoch 3: Train=0.5969, Val=0.8254
Epoch 4: Train=0.6355, Val=0.8582
Epoch 5: Train=0.6595, Val=0.8813
Epoch 6: Train=0.6835, Val=0.9060
Epoch 7: Train=0.7008, Val=0.9043
Epoch 8: Train=0.7104, Val=0.9170
Epoch 9: Train=0.7172, Val=0.9286
Epoch 10: Train=0.7274, Val=0.9267
Epoch 11: Train=0.7386, Val=0.9451
Epoch 12: Train=0.7435, Val=0.9393
Epoch 13: Train=0.7463, Val=0.9439
Epoch 14: Train=0.7518, Val=0.9451
Epoch 15: Train=0.7590, Val=0.9463
Epoch 16: Train=0.7582, Val=0.9490
Epoch 17: Train=0.7677, Val=0.9536
Epoch 18: Train=0.7748, Val=0.9534
Epoch 19: Train=0.7748, Val=0.9529
Epoch 20: Train=0.7782, Val=0.9565
Best Val Accuracy: 0.9565428502063608
Test Accuracy: 0.8982152816508645
CNN Parameters:  27193


After introducing dropout, both the training and validation accuracy decreased significantly, which could mean that the model is now underfitting the data so the model is not able to learn meaningful patterns from the training set. Because of this, I wanted to explore changing the activation function from Tanh to ReLU to improve the training efficiency.

In [119]:
class ReLUCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size = 3, padding=1)
    self.act1 = nn.ReLU()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.ReLU()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.ReLU()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    x = self.pool1(self.act1(self.conv1(x)))
    x = self.pool2(self.act2(self.conv2(x)))
    x = x.view(x.size(0), -1)
    x = self.fc2(self.act3(self.fc1(x)))

    return x


In [120]:
cnn_relu = ReLUCNN()
train_model(cnn_relu)
cnn_relu_test_accuracy = test_model(cnn_relu)
print("CNN Parameters: ", count_params(cnn_relu))

Epoch 1: Train=0.2629, Val=0.5169
Epoch 2: Train=0.5238, Val=0.7070
Epoch 3: Train=0.6515, Val=0.8024
Epoch 4: Train=0.7221, Val=0.8451
Epoch 5: Train=0.7748, Val=0.8781
Epoch 6: Train=0.8059, Val=0.8857
Epoch 7: Train=0.8323, Val=0.9252
Epoch 8: Train=0.8489, Val=0.9388
Epoch 9: Train=0.8613, Val=0.9429
Epoch 10: Train=0.8706, Val=0.9417
Epoch 11: Train=0.8804, Val=0.9612
Epoch 12: Train=0.8891, Val=0.9573
Epoch 13: Train=0.9008, Val=0.9655
Epoch 14: Train=0.9016, Val=0.9706
Epoch 15: Train=0.9119, Val=0.9774
Epoch 16: Train=0.9168, Val=0.9811
Epoch 17: Train=0.9184, Val=0.9794
Epoch 18: Train=0.9241, Val=0.9750
Epoch 19: Train=0.9247, Val=0.9879
Epoch 20: Train=0.9293, Val=0.9704
Best Val Accuracy: 0.9878611313425588
Test Accuracy: 0.9380925822643614
CNN Parameters:  27193


Replacing Tanh with ReLU decreased the model’s performance by a lot, which means that the activation function was not the main limiting factor in the model. This probably means that the issue is most likely related to the model’s capacity rather than its ability to optimize gradients and capture the complexity of the data. Because of this, I next increased the width of the CNN by adding more filters in the convolutional layers. A wider network can learn more features at each layer, which could be useful in this case.


In [121]:
class WideCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 32, kernel_size=3,padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(32,16, kernel_size=3,padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7,64)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(64,25)

  def forward(self,x):

    x = x.view(x.size(0), 1, 28, 28)
    x = self.pool1(self.act1(self.conv1(x)))
    x = self.pool2(self.act2(self.conv2(x)))
    x = x.view(x.size(0), -1)
    x = self.fc2(self.act3(self.fc1(x)))

    return x

In [122]:
cnn_wide = WideCNN()
train_model(cnn_wide)
cnn_wide_test_accuracy = test_model(cnn_wide)
print("CNN Parameters: ", count_params(cnn_wide))

Epoch 1: Train=0.5226, Val=0.8502
Epoch 2: Train=0.8278, Val=0.9507
Epoch 3: Train=0.9054, Val=0.9777
Epoch 4: Train=0.9390, Val=0.9925
Epoch 5: Train=0.9523, Val=0.9939
Epoch 6: Train=0.9628, Val=0.9976
Epoch 7: Train=0.9686, Val=0.9976
Epoch 8: Train=0.9745, Val=0.9934
Epoch 9: Train=0.9758, Val=0.9988
Epoch 10: Train=0.9811, Val=0.9990
Epoch 11: Train=0.9831, Val=0.9983
Epoch 12: Train=0.9826, Val=0.9995
Epoch 13: Train=0.9830, Val=1.0000
Epoch 14: Train=0.9822, Val=0.9998
Epoch 15: Train=0.9835, Val=1.0000
Epoch 16: Train=0.9873, Val=0.9998
Epoch 17: Train=0.9857, Val=0.9993
Epoch 18: Train=0.9884, Val=0.9993
Epoch 19: Train=0.9881, Val=0.9998
Epoch 20: Train=0.9884, Val=0.9995
Best Val Accuracy: 1.0
Test Accuracy: 0.9888455103179029
CNN Parameters:  56809


After increasing the width of the CNN, the test accuracy improved significantly compared to previous models, which means the original Base CNN did not have enough capacity to learn the complexity of the dataset, and adding more filters allowed the network to learn more. However, at the same time, there was a significant increase in the number of parameters used (almost double). Additionally, the model was able to reach a high accuracy in just a few epochs. Because of this, I wanted to explore whether increasing the depth of the network, instead of the width, could also help the model's performance since deeper models can capture more hierarchal feature representations.

In [123]:
class DeepCNN(nn.Module):

  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1,16,3,padding=1)
    self.act1 = nn.Tanh()

    self.conv2 = nn.Conv2d(16,16,3,padding=1)
    self.act2 = nn.Tanh()

    self.pool1 = nn.MaxPool2d(2)

    self.conv3 = nn.Conv2d(16,8,3,padding=1)
    self.act3 = nn.Tanh()

    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(8*7*7,32)
    self.fc2 = nn.Linear(32,25)

  def forward(self,x):

    x = x.view(x.size(0), 1, 28, 28)
    x = self.act1(self.conv1(x))
    x = self.act2(self.conv2(x))
    x = self.pool1(x)
    x = self.pool2(self.act3(self.conv3(x)))
    x = x.view(x.size(0), -1)
    x = self.fc2(self.fc1(x))

    return x

In [124]:
cnn_deep = DeepCNN()
train_model(cnn_deep)
cnn_deep_test_accuracy = test_model(cnn_deep)
print("CNN Parameters: ", count_params(cnn_deep))

Epoch 1: Train=0.3732, Val=0.7181
Epoch 2: Train=0.6875, Val=0.8628
Epoch 3: Train=0.8010, Val=0.9085
Epoch 4: Train=0.8499, Val=0.9429
Epoch 5: Train=0.8797, Val=0.9546
Epoch 6: Train=0.9038, Val=0.9728
Epoch 7: Train=0.9146, Val=0.9815
Epoch 8: Train=0.9273, Val=0.9852
Epoch 9: Train=0.9343, Val=0.9879
Epoch 10: Train=0.9406, Val=0.9859
Epoch 11: Train=0.9461, Val=0.9796
Epoch 12: Train=0.9474, Val=0.9893
Epoch 13: Train=0.9514, Val=0.9864
Epoch 14: Train=0.9537, Val=0.9930
Epoch 15: Train=0.9563, Val=0.9910
Epoch 16: Train=0.9583, Val=0.9891
Epoch 17: Train=0.9591, Val=0.9934
Epoch 18: Train=0.9613, Val=0.9913
Epoch 19: Train=0.9610, Val=0.9944
Epoch 20: Train=0.9629, Val=0.9905
Best Val Accuracy: 0.9944161204175771
Test Accuracy: 0.9643056330172894
CNN Parameters:  17041


The test accuracy of the deeper model was lower than that of the wider model, which meanst hat increasing the depth of the network did nothing to the performance of the model. Although the wider CNN achieved the highest test accuracy so far, this could be mainly because of the increased number of parameters.To address this and build a more parameter-efficient model, I introduced Global Average Pooling (GAP). Instead of flattening large feature maps, the GAP layer reduces each feature map to a smaller spatial representation, thereby decreasing the number of parameters in the fully connected layers.

In [125]:
class GAPCNN(nn.Module):

  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1,8,3,padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8,16,3,padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.gap = nn.AdaptiveAvgPool2d((3,3))

    self.fc1 = nn.Linear(16*3*3,32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32,25)

  def forward(self,x):

    x = x.view(x.size(0),1,28,28)
    x = self.pool1(self.act1(self.conv1(x)))
    x = self.pool2(self.act2(self.conv2(x)))
    x = self.gap(x)
    x = x.view(x.size(0),-1)
    x = self.fc2(self.act3(self.fc1(x)))

    return x

In [126]:
cnn_gap = GAPCNN()
train_model(cnn_gap, 20)
cnn_gap_test_accuracy = test_model(cnn_gap)
print("CNN Parameters:", count_params(cnn_gap))

Epoch 1: Train=0.2083, Val=0.3880
Epoch 2: Train=0.4585, Val=0.5948
Epoch 3: Train=0.6162, Val=0.7094
Epoch 4: Train=0.7009, Val=0.7973
Epoch 5: Train=0.7481, Val=0.8369
Epoch 6: Train=0.7823, Val=0.8310
Epoch 7: Train=0.8067, Val=0.8769
Epoch 8: Train=0.8283, Val=0.8939
Epoch 9: Train=0.8390, Val=0.9041
Epoch 10: Train=0.8539, Val=0.9133
Epoch 11: Train=0.8657, Val=0.9211
Epoch 12: Train=0.8738, Val=0.9128
Epoch 13: Train=0.8816, Val=0.9184
Epoch 14: Train=0.8909, Val=0.9289
Epoch 15: Train=0.8953, Val=0.9432
Epoch 16: Train=0.9027, Val=0.9405
Epoch 17: Train=0.9073, Val=0.9524
Epoch 18: Train=0.9124, Val=0.9536
Epoch 19: Train=0.9130, Val=0.9541
Epoch 20: Train=0.9188, Val=0.9590
Best Val Accuracy: 0.958970623937849
Test Accuracy: 0.9124372559955382
CNN Parameters: 6713


While the GAP model significantly reduced the number of parameters, it also resulted in a drop in accuracy. To try to improve performance without increasing model size, I tried to implement a residual connection in the GAP CNN because residual connections help the network be able learn and identify networks mappings more quickly, increasing training efficiency without adding many extra parameters because the residual adds an element wise addition instead of convolutional layers, thereby decreasing the amount of extra parameters.



In [127]:
class ResidualGAPCNN(nn.Module):
  
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 8, 3, padding = 1)
    self.bn1 = nn.BatchNorm2d(8)
    self.act1 = nn.Tanh()

    self.conv2 = nn.Conv2d(8, 8, 3, padding = 1)
    self.bn2 = nn.BatchNorm2d(8)
    self.act2 = nn.Tanh()
    self.pool = nn.MaxPool2d(2)

    self.conv3 = nn.Conv2d(8, 16, 3, padding = 1)
    self.bn3 = nn.BatchNorm2d(16)
    self.act3 = nn.Tanh()

    self.gap = nn.AdaptiveAvgPool2d((3,3))
    self.gmp = nn.AdaptiveMaxPool2d((3,3))

    self.fc1 = nn.Linear(16*3*3*2, 32)
    self.act_fc = nn.Tanh()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    out1 = self.act1(self.bn1(self.conv1(x)))
    out2 = self.act2(self.bn2(self.conv2(out1))) + out1
    out2 = self.pool(out2)
    out3 = self.act3(self.bn3(self.conv3(out2)))
    out3 = self.pool(out3)
    gap = self.gap(out3)
    gmp = self.gmp(out3)
    x = torch.cat([gap, gmp], dim = 1)
    x = x.view(x.size(0), -1)
    x = self.act_fc(self.fc1(x))
    x = self.fc2(x)

    return x


In [128]:
cnn_residual_gap = ResidualGAPCNN()
train_model(cnn_residual_gap)
cnn_res_gap_test_accuracy = test_model(cnn_residual_gap)
print("CNN Parameters:", count_params(cnn_residual_gap))

Epoch 1: Train=0.3901, Val=0.5887
Epoch 2: Train=0.6857, Val=0.7696
Epoch 3: Train=0.7836, Val=0.8597
Epoch 4: Train=0.8348, Val=0.9058
Epoch 5: Train=0.8665, Val=0.9245
Epoch 6: Train=0.8872, Val=0.9437
Epoch 7: Train=0.9096, Val=0.9408
Epoch 8: Train=0.9197, Val=0.9626
Epoch 9: Train=0.9301, Val=0.9718
Epoch 10: Train=0.9382, Val=0.9723
Epoch 11: Train=0.9450, Val=0.9842
Epoch 12: Train=0.9535, Val=0.9879
Epoch 13: Train=0.9562, Val=0.9767
Epoch 14: Train=0.9627, Val=0.9840
Epoch 15: Train=0.9643, Val=0.9903
Epoch 16: Train=0.9670, Val=0.9903
Epoch 17: Train=0.9687, Val=0.9908
Epoch 18: Train=0.9716, Val=0.9840
Epoch 19: Train=0.9726, Val=0.9917
Epoch 20: Train=0.9750, Val=0.9898
Best Val Accuracy: 0.99174556931294
Test Accuracy: 0.9581706636921361
CNN Parameters: 11969


Adding the residual connections to the GAP CNN model really help improve the validation and test accuracies while still keeping the parameter count lower than the previous Wide CNN models. To continue to improve generalization, I tried data augmentation by adding small rotations and translations before the model.

In [134]:
from torchvision import transforms

class AugmentationSignLanguageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].reshape(28, 28).astype(np.uint8)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform:
            image = self.transform(image)
        else:
            image = torch.tensor(image, dtype=torch.float32) / 255.0
            image = image.unsqueeze(0)

        return image, label

In [135]:
augmentation_strategy = transforms.Compose([
    transforms.ToTensor(),
    # Randomly rotate the image by up to 10 degrees in either direction
    transforms.RandomRotation(degrees=10), 
    # Randomly translate/shift the image vertically or horizontally by up to 10%
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)) 
])

# For testing/validation, we ONLY convert to tensor. We NEVER augment test data.
base_transform = transforms.Compose([
    transforms.ToTensor()
])

aug_train_dataset = AugmentationSignLanguageDataset(
    train_imgs, train_lbls, transform=augmentation_strategy
)

aug_train_loader = DataLoader(aug_train_dataset, batch_size=64, shuffle=True)

In [136]:
def augmented_train_model(model, val_loader, num_epochs=20):
  optimizer = optim.Adam(model.parameters(), lr=0.001)
  loss_function = nn.CrossEntropyLoss()

  for epoch in range(num_epochs):
    model.train()

    train_correct = 0
    train_total = 0

    for imgs, labels in aug_train_loader:
      preds = model(imgs)
      loss = loss_function(preds, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      predicted = preds.argmax(dim=1)
      train_correct += (predicted == labels).sum().item()
      train_total += labels.size(0)

    train_acc = train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
      for imgs, labels in val_loader:
        preds = model(imgs)
        predicted = preds.argmax(dim=1)
        val_correct += (predicted == labels).sum().item()
        val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}: Train Accuracy={train_acc:.4f}, Val Accuracy={val_acc:.4f}")

In [139]:
cnn_gap_aug = ResidualGAPCNN()
augmented_train_model(cnn_gap_aug, val_loader, num_epochs=20)
cnn_test_accuracy_aug = test_model(cnn_gap_aug)
print("CNN Parameters:", count_params(cnn_gap_aug))

Epoch 1: Train Accuracy=0.3279, Val Accuracy=0.3918
Epoch 2: Train Accuracy=0.6289, Val Accuracy=0.5147
Epoch 3: Train Accuracy=0.7539, Val Accuracy=0.6089
Epoch 4: Train Accuracy=0.8211, Val Accuracy=0.6497
Epoch 5: Train Accuracy=0.8640, Val Accuracy=0.6400
Epoch 6: Train Accuracy=0.8915, Val Accuracy=0.7259
Epoch 7: Train Accuracy=0.9133, Val Accuracy=0.7286
Epoch 8: Train Accuracy=0.9266, Val Accuracy=0.7378
Epoch 9: Train Accuracy=0.9372, Val Accuracy=0.7771
Epoch 10: Train Accuracy=0.9459, Val Accuracy=0.7512
Epoch 11: Train Accuracy=0.9497, Val Accuracy=0.7655
Epoch 12: Train Accuracy=0.9569, Val Accuracy=0.8070
Epoch 13: Train Accuracy=0.9592, Val Accuracy=0.7791
Epoch 14: Train Accuracy=0.9626, Val Accuracy=0.7618
Epoch 15: Train Accuracy=0.9650, Val Accuracy=0.8152
Epoch 16: Train Accuracy=0.9721, Val Accuracy=0.8111
Epoch 17: Train Accuracy=0.9698, Val Accuracy=0.8230
Epoch 18: Train Accuracy=0.9736, Val Accuracy=0.8208
Epoch 19: Train Accuracy=0.9722, Val Accuracy=0.8434
Ep

This first augmentation strategy did not work as well because it consisted of relatively strong transformations (rotations of 10 degree and translations of 10 degrees of image size). The testing accuracy went really low because many of these signs migh have subtle differences in finger positions, so these transformations may have distorted any patterns significantly, making it harder for the model to learn meaningful features. Because of this, the second augmentation uses smaller rotations and translations (5 degrees), which help introduce variability in the training data, but also preserve the shapes of the signs.

In [140]:
def augmented_train_model2(model, val_loader, num_epochs=20, lr = 0.001):
  optimizer = optim.Adam(model.parameters(), lr = lr)
  loss_function = nn.CrossEntropyLoss()

  best_val_acc = 0
  train_acc_list, val_acc_list = [], []

  for epoch in range(num_epochs):
    model.train()
    train_correct = 0
    train_total = 0

    for imgs, labels in aug_train_loader:
      preds = model(imgs)
      loss = loss_function(preds, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      predicted = preds.argmax(dim=1)
      train_correct += (predicted == labels).sum().item()
      train_total += labels.size(0)

    train_acc = train_correct / train_total
    train_acc_list.append(train_acc)

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
      for imgs, labels in val_loader:
        preds = model(imgs)
        predicted = preds.argmax(dim = 1)
        val_correct += (predicted == labels).sum().item()
        val_total += labels.size(0)

    val_acc = val_correct / val_total
    val_acc_list.append(val_acc)


    if val_acc > best_val_acc:
      best_val_acc = val_acc
      best_weights = model.state_dict()

    print(f"Epoch {epoch+1}: Train Accuracy={train_acc:.4f}, Val Accuracy={val_acc:.4f}")

  model.load_state_dict(best_weights)
  print("Best Val Accuracy:", best_val_acc)
  return train_acc_list, val_acc_list


In [141]:
cnn_gap_aug2 = ResidualGAPCNN()  
augmented_train_model2(cnn_gap_aug2, val_loader, num_epochs=20)
cnn_test_accuracy_aug = test_model(cnn_gap_aug2)
print("CNN Parameters:", count_params(cnn_gap_aug2))


Epoch 1: Train Accuracy=0.3277, Val Accuracy=0.3251
Epoch 2: Train Accuracy=0.6346, Val Accuracy=0.5266
Epoch 3: Train Accuracy=0.7588, Val Accuracy=0.6310
Epoch 4: Train Accuracy=0.8231, Val Accuracy=0.6900
Epoch 5: Train Accuracy=0.8604, Val Accuracy=0.7397
Epoch 6: Train Accuracy=0.8895, Val Accuracy=0.7572
Epoch 7: Train Accuracy=0.9089, Val Accuracy=0.7652
Epoch 8: Train Accuracy=0.9182, Val Accuracy=0.8228
Epoch 9: Train Accuracy=0.9315, Val Accuracy=0.8009
Epoch 10: Train Accuracy=0.9407, Val Accuracy=0.7560
Epoch 11: Train Accuracy=0.9478, Val Accuracy=0.7786
Epoch 12: Train Accuracy=0.9519, Val Accuracy=0.7810
Epoch 13: Train Accuracy=0.9581, Val Accuracy=0.8165
Epoch 14: Train Accuracy=0.9618, Val Accuracy=0.8002
Epoch 15: Train Accuracy=0.9640, Val Accuracy=0.8203
Epoch 16: Train Accuracy=0.9663, Val Accuracy=0.8526
Epoch 17: Train Accuracy=0.9690, Val Accuracy=0.8196
Epoch 18: Train Accuracy=0.9719, Val Accuracy=0.8599
Epoch 19: Train Accuracy=0.9729, Val Accuracy=0.8679
Ep

This was able to significantly improve test accuracy compared to the previous agumentation. To increase it more, I refined my training process more by using more milder augmentations to introduce some variability but also avoiding distorting the images too much. I also combined both the original dataset and the augmented one durng training to allow the model to learn from the true distribution.  

In [151]:
mild_aug_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(3),          
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05))  
])

base_transform = transforms.Compose([
    transforms.ToTensor()
])

In [152]:
aug_train_dataset2 = AugmentationSignLanguageDataset(train_imgs, train_lbls, transform=mild_aug_transform)
original_train_dataset = SignLanguageDataset(train_imgs, train_lbls, transform=base_transform)

#combine original and augmented datasets
combined_train_dataset = ConcatDataset([original_train_dataset, aug_train_dataset2])
combined_train_loader = DataLoader(combined_train_dataset, batch_size=64, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=64)

In [153]:
def augmented_train_model3(model, val_loader, num_epochs=30, lr=0.001):
  optimizer = optim.Adam(model.parameters(), lr=lr)
  loss_function = nn.CrossEntropyLoss()
  
  best_val_acc = 0
  train_acc_list, val_acc_list = [], []

  for epoch in range(num_epochs):
    model.train()
    train_correct, train_total = 0, 0

    for imgs, labels in combined_train_loader:
      preds = model(imgs)
      loss = loss_function(preds, labels)
      
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      
      predicted = preds.argmax(dim=1)
      train_correct += (predicted == labels).sum().item()
      train_total += labels.size(0)

    train_acc = train_correct / train_total
    train_acc_list.append(train_acc)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
      for imgs, labels in val_loader:
        preds = model(imgs)
        predicted = preds.argmax(dim=1)
        val_correct += (predicted == labels).sum().item()
        val_total += labels.size(0)

    val_acc = val_correct / val_total
    val_acc_list.append(val_acc)

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      best_weights = model.state_dict()

    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

  model.load_state_dict(best_weights)
  print("Best Val Accuracy:", best_val_acc)

  return train_acc_list, val_acc_list

In [154]:
cnn_gap_aug3 = ResidualGAPCNN()
augmented_train_model3(cnn_gap_aug3, val_loader, num_epochs=20, lr=0.001)
cnn_test_accuracy_aug = test_model(cnn_gap_aug3)
print("CNN Parameters:", count_params(cnn_gap_aug3))

Epoch 1: Train Acc=0.6242, Val Acc=0.7101
Epoch 2: Train Acc=0.9226, Val Acc=0.8497
Epoch 3: Train Acc=0.9763, Val Acc=0.8602
Epoch 4: Train Acc=0.9886, Val Acc=0.9284
Epoch 5: Train Acc=0.9934, Val Acc=0.9395
Epoch 6: Train Acc=0.9955, Val Acc=0.9226
Epoch 7: Train Acc=0.9972, Val Acc=0.9597
Epoch 8: Train Acc=0.9976, Val Acc=0.9442
Epoch 9: Train Acc=0.9979, Val Acc=0.9675
Epoch 10: Train Acc=0.9972, Val Acc=0.8971
Epoch 11: Train Acc=0.9984, Val Acc=0.9614
Epoch 12: Train Acc=0.9986, Val Acc=0.9456
Epoch 13: Train Acc=0.9991, Val Acc=0.9621
Epoch 14: Train Acc=0.9987, Val Acc=0.9330
Epoch 15: Train Acc=0.9991, Val Acc=0.9240
Epoch 16: Train Acc=0.9985, Val Acc=0.9053
Epoch 17: Train Acc=0.9987, Val Acc=0.9585
Epoch 18: Train Acc=0.9988, Val Acc=0.9726
Epoch 19: Train Acc=0.9988, Val Acc=0.9910
Epoch 20: Train Acc=0.9992, Val Acc=0.9743
Best Val Accuracy: 0.9910172371934935
Test Accuracy: 0.9290295593976575
CNN Parameters: 11969
